# שיעור 18: אבטחת סוכני AI עם קבלות קריפטוגרפיות

## פנקס אימון מעשי

פנקס זה עובר דרך ארבעה תרגילים:

1. **חתום על הקבלה הראשונה שלך** עבור קריאת כלי סוכן ואמת אותה.
2. **זייף את הקבלה** וצפה באימות שנכשל.
3. **בנה שרשרת שלוש קבלות** ואשר את שלמות השרשרת.
4. **עטוף קריאת כלי במסגרת סוכן של Microsoft** כך שכל פעולה תפיק קבלה.

כל הפונקציות הקריפטוגרפיות מיובאות מספריות מתוחזקות היטב (`pynacl` עבור Ed25519, `jcs` עבור JSON קנוני לפי RFC 8785, `hashlib` מספריית הסטנדרט של פייתון עבור SHA-256). לוגיקת הקבלה עצמה היא פייתון פשוט שאתה יכול לקרוא ולשנות.

הפעל את התאים לפי הסדר. כל חלק קצר ומנותק.


## התקנה

התקן את שתי התלויות. לשתיהן יש רשיונות מתירים (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## כלי עזר

שני כלי העזר האלה מטפלים בקידוד base64url (ללא מילוי) וביצירת HASH מסוג SHA-256 של אובייקטים אקראיים. הם שומרים על שאר המחברת ממוקדת בלוגיקת הקבלה עצמה.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## חלק 1: חתום על הקבלה הראשונה שלך

דמיין שהסוכן שלנו עבור **Contoso Travel** רק בדק טיסות מסידני ללוס אנג'לס עבור לקוח. אנחנו רוצים לרשום את השיחה על הכלי הזה כהסכמה חתומה כך שמבקר עתידי יוכל לאמת אותה ללא צורך לסמוך עלינו.

### שלב 1.1: יצירת מפתח חתימה

בסביבת ייצור, מפתח החתימה של הסוכן היה מאוחסן במודול אבטחת חומרה (HSM), Azure Key Vault, או מחסן מוגן דומה. בשיעור זה אנו מייצרים מפתח טרי בזיכרון.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### שלב 1.2: בניית המטען של הקבלה

המטען מכיל את כל מה שאנחנו רוצים שהקבלה תעיד עליו: מי פעל, איזה כלי, עם אילו ארגומנטים, מה התקבל, תחת איזו מדיניות, ומתי. אנו מבצעים גיבוב של הארגומנטים והתוצאה במקום לכלול אותם במפורש כדי שהקבלה לא תחשוף תוכן רגיש.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### שלב 1.3: חתום והרכיב את הקבלה

שלושה שלבים:

1. עקב את המטען באמצעות JCS כך ששתי מימושים שמייצרים את אותה הקבלה הלוגית ייצרו בתים זהים בדיוק.
2. חשב את הגיבוב של בתים המקוריים עם SHA-256.
3. חתום את הגיבוב עם מפתח פרטי Ed25519.

החתימה מצורפת אז למטען המקורי כדי לייצר את הקבלה הסופית.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### שלב 1.4: אמת את הקבלה

האימות הופך את התהליך. אנו מסירים את החתימה, מחשבים מחדש את הקוד הקנוני, ובודקים את החתימה מול המפתח הציבורי בקבלה.

מבקר המבצע את האימות הזה אינו זקוק מאיתנו לשום דבר פרט לקבלה עצמה. אין צורך לקרוא לשירות, לשאול ספריית מפתחות או לבקש אמון.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

עליך לראות `Receipt is valid: True`. הסוכן הפיק את רישום הביקורת החתום קריפטוגרפית הראשון שלו.


## חלק 2: לזייף את הקבלה

כל המטרה של קבלות היא שהן ניתנות לזיהוי במקרה של זיוף. בואו נוכיח את זה.

נשנה בדיוק תו אחד של הקבלה ונצפה באי הצלחת האימות.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### מה בדיוק קרה?

כששינינו את `policy_id`, בתים הקנוניים השתנו. ה-hash מסוג SHA-256 של אותם בתים השתנה. החתימה (שהייתה על ה-hash המקורי) כבר לא תואמת ל-hash החדש. האימות מחזיר נכון `False`.

אין דרך לשנות כל שדה בקבלה ועדיין לאמת אותה, אלא אם לתוקף יש את המפתח הפרטי. כל עוד המפתח הפרטי נמצא במאגר מפתחות והמפתח הציבורי פורסם, זיופים בלתי אפשריים להסתיר.

נסה זאת בעצמך: שנה את `tool_name` או `agent_id` או `timestamp` בתא למעלה והריץ מחדש. כל שינוי מייצר קבלה לא תקפה.


## סעיף 3: שרשרת קבלות יחד

קבלה יחידה מגן על פעולה בודדת. רוב הסוכנים מבצעים פעולות רבות. כדי להפוך את כל הרצף לעמיד למניפולציות, אנו מקשרים כל קבלה לזו הקודמת על ידי הכללת תג ה-קבלה הקודמת בתוך המטען של הקבלה החדשה.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

אם מישהו מסיר או מסדר מחדש קבלה, השרשרת נשברת בנקודה המדויקת הזו. אימות של כל קבלה מאוחרת יותר נכשל כיוון ש-`previous_receipt_hash` שלה כבר לא תואם לתג האמיתי של הקבלה הקודמת לה.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

עכשיו שברו את השרשרת על ידי טלטול הקבלה האמצעית ואמתו מחדש. הקבלה המטולטלת נכשלת בבדיקת החתימה שלה, וחשבון הקבלה הבא נכשל בבדיקת הקישור בשרשרת (מכיוון ש-`previous_receipt_hash` שלה כבר לא תואם את ה-hash של הקבלה האמצעית ששונתה).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

חשבונית 0 עדיין מאומתת (לא שונתה ואין לה קודמת להסתמך עליה). חשבונית 1 נכשלת בבדיקת החתימה שלה כי שינינו את `tool_args_hash`. חשבונית 2 נכשלת בבדיקת הקישור בשרשרת כי ה-`previous_receipt_hash` שלה חושב אל מול חשבונית 1 המקורית (כעת שונתה).

אפילו אם תוקף יחתום מחדש על חשבונית 1 ששונתה (שהוא לא יכול לעשות ללא המפתח הפרטי), חוסר ההתאמה בקישור בשרשרת בחשבונית 2 עדיין יחשוף את השיבוש. כדי להסתיר את השינוי, התוקף יצטרך לחתום מחדש על כל החשבוניות מהנקודה שבה השינוי התרחש ואילך, דבר שדורש החזקת המפתח הפרטי.


## סעיף 4: לעטוף קריאת כלי סוכן עם חתימת קבלה

בפריסה אמיתית, אינך רוצה שכל כותב סוכן יזכור לקרוא ל-`make_receipt`. אתה רוצה שחתימת הקבלה תהיה אוטומטית עבור כל קריאת כלי.

הנה הדפוס הפשוט ביותר: מחלקת עטיפה שמקבלת כל פונקציית כלי הניתנת לקריאה ומחזירה גרסה שמפיקה קבלה. זה מתאים לכל מסגרת סוכן, כולל Microsoft Agent Framework (`agent_framework.foundry`).

אם אין לך פרויקט Microsoft Foundry מוגדר, ההדגמה המקומית למטה עדיין מציגה את הדפוס.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### אינטגרציה עם מסגרת הסוכן של מיקרוסופט

המעטפת `ReceiptedTool` שלמעלה-מטה אינה תלויה במסגרת. כדי להשתמש בה בתוך סוכן שנבנה עם מסגרת הסוכן של מיקרוסופט, רישום הפונקציה עטופה ככלי. קונספט (תחליפו את הדמה ברישום אמיתי של כלי Foundry של מיקרוסופט):

```python
# קוד פסאודו שמראה את צורת האינטגרציה.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="אתה סוכן טיולים של Contoso ...",
#     tools=[receipted_lookup],   # הכלי עטוף, לא הפונקציה הגולמית
# )
# response = agent.run("מצא טיסות מסידני ללוס אנג'לס ביוני.")
#
# # לאחר ההרצה, לכל קריאה לכלי שבוצעה על ידי הסוכן יש קבלה חתומה:
# audit_chain = receipted_lookup.receipts
```

מסגרת הסוכן אינה צריכה לדעת כלום על קבלות. חתימת הקבלה עטופה סביב הכלי, לא מובנית למסגרת. כך מוסיפים ייחוס לקוד סוכן קיים מבלי לכתוב מחדש את הסוכן.


## סיכום ואתגר מתיחה

יש לך:

- יצרת זוג מפתחות Ed25519.
- בנית וחתמת על קבלה עבור קריאת כלי סוכן.
- אימתת את הקבלה באופן לא מקוון באמצעות המפתח הציבורי בלבד.
- זייפת קבלה וצפת בכשל באימות.
- בנית רצף שרשור גיבוב של שלוש קבלות.
- זייפת באמצע השרשור וצפת בכישלון בחותמת ובקישור בשרשרת.
- עטפת פונקציה של כלי עם חתימת קבלה אוטומטית.

**אתגר מתיחה.** הרחב את סכמת הקבלה עם שדה `request_id` (UUID למעקב מופץ). עדכן את `make_receipt` לכלול אותו, ואשר שהקבלות עדיין מאומתות מקצה לקצה. לאחר מכן שנה את השדה לאחר החתימה ואשר שהאימות נכשל. זה מאלץ אותך להבין כיצד כל בת בביטוי הקנוני תורם לחתימה.

**גבול חשוב.** הקבלות מוכיחות שלושה דברים בלבד: שיוך (המפתח הזה חתם על התוכן הזה), שלמות (התוכן לא שונה מאז החתימה), וסדר (קבלה זו הגיעה אחרי קבלה אחרת). הן אינן מוכיחות שהפעולה של הסוכן הייתה תקינה, שהמדיניות בשם `policy_id` אכן הוערכה, או שהסוכן עקב אחרי כל הכללים. הקבלות הן בסיס. המשילות היא המערכת שאתה בונה מעל.

קרא שוב את קובץ ה-README של השיעור עם הגבול הזה בראש. הטעות הנפוצה ביותר שצוותים עושים עם קבלות היא להניח "יש לנו קבלות" פירושו "אנחנו נשלטים." זה לא נכון. קבלות מאפשרות לבצע ביקורת של התנהגות הסוכן. הן לא מוודאות שהיא נכונה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
